# Chapter 1 &mdash; The Hierarchy, Re-derived from C Programs

**Concept 15 of the Chapter 1 decomposition:** *Machine Classes as Programming Restrictions*

Forget tapes. Each machine class is what you get by restricting how a program may <b>allocate and access memory</b>.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Machines-As-Programming-Restrictions/Concept-Machines-As-Programming-Restrictions.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


* **Finite automata** &mdash; only finitely many *finite* variables (a bit, a char).
  No heap. No recursion.
* **Pushdown automata** &mdash; add functions that may call each other **recursively**.
  The call stack *is* the PDA's stack.
* **Turing machines** &mdash; add unbounded memory, freely accessed.
* **Linear bounded automata** &mdash; unbounded memory **snipped** to the input length.

All four differ only in **how memory may be allocated or accessed**.

## 2. Definitions

### FA-style: finitely many finite variables

Parity of `1`s. One boolean. No matter how long the input, memory never grows.

In [ ]:
def fa_style_parity(s):
    odd = False                  # ONE finite variable -- this is the whole memory
    for ch in s:
        if ch == '1':
            odd = not odd
    return odd

### PDA-style: recursion, but each frame holds only finite data

The recursion *stack* is unbounded; each frame is not.

In [ ]:
import sys
sys.setrecursionlimit(10000)

def pda_style_balanced(s, i=0, depth=0):
    """Recursive descent. The call stack does the counting."""
    if i == len(s):
        return depth == 0
    if s[i] == '(':
        return pda_style_balanced(s, i+1, depth+1)
    if depth == 0:
        return False             # a ')' with nothing open
    return pda_style_balanced(s, i+1, depth-1)

### TM-style: unbounded, freely-accessed memory

An array with no a priori bound, indexed in any order. This is what buys $ww$.

In [ ]:
def tm_style_is_ww(s):
    """Uses an unbounded, randomly-accessed buffer -- TM power."""
    if len(s) % 2:
        return False
    tape = list(s)               # unbounded, freely indexed
    half = len(tape) // 2
    for k in range(half):        # jump anywhere on the tape
        if tape[k] != tape[half + k]:
            return False
    return True

## 3. Tests

FA-style: memory usage is **constant** however long the input.

In [ ]:
for s in ['', '1', '11', '10101', '1'*999]:
    print("len %-4d parity odd? %s" % (len(s), fa_style_parity(s)))
print()
print("One boolean handled a 999-character input. No growth. That is an FA.")

PDA-style: the recursion depth **grows with the nesting**.

In [ ]:
for s in ['', '()', '(())', '((()))', '(()', ')(']:
    print("%-10s balanced? %s" % (repr(s), pda_style_balanced(s)))

TM-style: $ww$, which neither of the previous two could manage.

In [ ]:
for s in ['abab', 'abba', '', 'aa', 'abc']:
    print("%-6s is w w? %s" % (repr(s), tm_style_is_ww(s)))
assert tm_style_is_ww("abab") and not tm_style_is_ww("abba")
print()
print("Note tm_style_is_ww jumps to tape[half+k] -- RANDOM ACCESS.")
print("A stack cannot do that; that is exactly why w w needs more than a PDA.")

## 4. Exercises


1. Rewrite `pda_style_balanced` as a loop with an explicit list used only via
   append/pop. You have just written a PDA by hand.
2. Modify it to *peek* inside the stack (not just the top). Which restriction have
   you broken, and what power did you just gain?
3. `tm_style_is_ww` allocates a list as long as the input. Is that an LBA or a full TM?

In [ ]:
# Your work for the exercises above.